# AIOps & MLOps — Churn modell

> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)

Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.
Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.

## Hogyan futtasd

```bash
# 1. Virtuális környezet (Python 3.10+)
python -m venv .venv

# Windows:
.venv\Scripts\activate

# macOS/Linux:
source .venv/bin/activate

# 2. Telepítsd a függőségeket (a notebook első cellája)

# 3. Indítsd a Jupytert
jupyter lab
# vagy
jupyter notebook
```

Minden cella saját magában értelmezhető. A `# %%` kommentek Jupyterben és VS Code-ban is a cellák határát jelölik.


## 1. Környezet


In [ ]:
%pip install mlflow scikit-learn pandas --quiet

## 2. Szintetikus churn adat


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'tenure_months': np.random.randint(1, 60, n),
    'monthly_charges': np.random.uniform(20, 120, n),
    'total_orders': np.random.poisson(10, n),
    'days_since_last': np.random.exponential(30, n).astype(int),
    'support_tickets': np.random.poisson(2, n),
})

# Churn valószínűség: inaktív + kevés rendelés → churn.
prob = 1 / (1 + np.exp(-(
    0.05 * df['days_since_last']
    - 0.1 * df['total_orders']
    + 0.2 * df['support_tickets']
    - 2.0
)))
df['churn'] = (np.random.random(n) < prob).astype(int)

print(df.head())
print(f'\nChurn arány: {df["churn"].mean():.1%}')


## 3. Modell tanítás MLflow tracking-gel


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

mlflow.set_experiment('webshop/churn-model')

X = df.drop(columns=['churn'])
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for n_est in [50, 100, 200]:
    with mlflow.start_run(run_name=f'rf_{n_est}'):
        mlflow.log_param('model', 'RandomForest')
        mlflow.log_param('n_estimators', n_est)

        model = RandomForestClassifier(n_estimators=n_est, random_state=42)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        metrics = {
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred),
            'recall': recall_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, proba),
        }

        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, 'model')
        print(f'n_est={n_est} → acc={metrics["accuracy"]:.3f}, auc={metrics["roc_auc"]:.3f}')


## 4. Model Registry — staging promotion


In [ ]:
# Legjobb run regisztrálása
client = mlflow.tracking.MlflowClient()
best_run = client.search_runs(
    experiment_ids=[mlflow.get_experiment_by_name('webshop/churn-model').experiment_id],
    order_by=['metrics.roc_auc DESC'],
    max_results=1,
)[0]

print(f'Best run: {best_run.info.run_id}, AUC={best_run.data.metrics["roc_auc"]:.3f}')

# Regisztrálás
mv = mlflow.register_model(
    model_uri=f'runs:/{best_run.info.run_id}/model',
    name='webshop-churn',
)
print(f'\nRegistered as version: {mv.version}')

# Staging promotion (alias használata a régi "stage" helyett)
client.set_registered_model_alias('webshop-churn', 'staging', mv.version)
print('Staging alias beállítva')


## 5. Fast

API serving minta (kódként)


In [ ]:
SERVING_CODE = '''
# app.py
from fastapi import FastAPI
from pydantic import BaseModel
import mlflow

app = FastAPI()
model = mlflow.pyfunc.load_model('models:/webshop-churn@staging')

class ChurnRequest(BaseModel):
    tenure_months: int
    monthly_charges: float
    total_orders: int
    days_since_last: int
    support_tickets: int

@app.post("/predict")
def predict(req: ChurnRequest):
    import pandas as pd

    df = pd.DataFrame([req.model_dump()])
    prob = float(model.predict(df)[0])
    return {"churn_probability": prob, "churn_flag": prob > 0.5}

# Futtatás:
#   uvicorn app:app --host 0.0.0.0 --port 8000
# Tesztelés:
#   curl -X POST http://localhost:8000/predict \
#     -H "Content-Type: application/json" \
#     -d '{"tenure_months":5,"monthly_charges":80,"total_orders":3,"days_since_last":45,"support_tickets":4}'
'''

print(SERVING_CODE)


## 6. Drift detekció — Kolmogorov-Smirnov teszt


In [ ]:
from scipy import stats

# Szimuláljunk driftet: a days_since_last eloszlása megváltozott.
current_days = df['days_since_last'].values
drifted_days = np.random.exponential(60, 500).astype(int)  # duplájára nőtt átlag

ks_stat, p_value = stats.ks_2samp(current_days, drifted_days)

print(f'KS statisztika: {ks_stat:.3f}')
print(f'p-value:        {p_value:.4f}')
print(f'Drift? {"IGEN — szignifikáns" if p_value < 0.05 else "NEM"}')


## Következő lépések

- Térj vissza a [web-alapú kurzushoz](./index.html) a teljes anyagért, diagramokért és kvízekért.
- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.
- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)

---

*Engineering Crash Courses · MIT License · Magyar Data & AI Engineering kurzusok*
